In [1]:
!pip install git+https://github.com/mittagessen/kraken@main

  Cloning https://github.com/mittagessen/kraken (to revision main) to /tmp/pip-req-build-s5qrsgnk
  Running command git clone --filter=blob:none --quiet https://github.com/mittagessen/kraken /tmp/pip-req-build-s5qrsgnk
  Resolved https://github.com/mittagessen/kraken to commit 1e4841c7b7d43fbb914455fa3d67728dff760a4b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!kraken get 10.5281/zenodo.2577813

Processing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 2.9/2.9 MB 0:00:00 0:00:02
Model dir: /root/.local/share/htrmopo/c895582c-4416-5715-a3e1-4ff1fb2766a6 (model files: en_best.mlmodel)


In [3]:
!find / -name "*.mlmodel" 2>/dev/null

/usr/local/lib/python3.12/dist-packages/kraken/blla.mlmodel
/root/.local/share/htrmopo/c895582c-4416-5715-a3e1-4ff1fb2766a6/en_best.mlmodel


In [4]:
!pip show kraken

Name: kraken
Version: 5.3.1.dev129
Summary: OCR/HTR engine for all the languages
Home-page: https://kraken.re
Author: Benjamin Kiessling
Author-email: mittagessen@l.unchti.me
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: click, coremltools, htrmopo, iso639-lang, jinja2, jsonschema, lightning, lxml, numpy, Pillow, platformdirs, protobuf, pyarrow, python-bidi, regex, requests, rich, scikit-image, scikit-learn, scipy, shapely, threadpoolctl, torch, torchmetrics, torchvision
Required-by: 


In [ ]:
# Add your own path from you drive
# from google.colab import drive
# drive.mount('/content/drive1')

Mounted at /content/drive1


In [7]:
import os
import torch
import warnings
from tqdm import tqdm
from PIL import Image
from kraken import binarization, pageseg, rpred
from kraken.lib import models, segmentation

In [ ]:
image_folder = "/content/drive1/MyDrive/iamges"
list_file = "/content/list_of_files.txt"    # this is a file containing the names of the images that have ground truth text file(extracted words)
output_folder = "/content/drive1/MyDrive/ocr_results_kraken"
model_path = '/root/.local/share/htrmopo/c895582c-4416-5715-a3e1-4ff1fb2766a6/en_best.mlmodel'  # change this part according to the place that your kraken model is saved with the command written above '!find / -name "*.mlmodel" 2>/dev/null'

In [9]:
warnings.filterwarnings("ignore")

In [ ]:
os.makedirs(output_folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.load_any(model_path)
if model.nn is None:
    model.load_model()

with open(list_file, "r", encoding="utf-8") as f:
    raw_names = [line.strip() for line in f if line.strip()]

image_names = []
for name in raw_names:
    if name.lower().endswith(".txt"):
        name = name[:-4]
    if not name.lower().endswith(('.jpg', '.png', '.jpeg', '.tiff')):
        name = name + ".jpg"
    image_names.append(name)

print(f"Found {len(image_names)} images to process.")

for img_name in tqdm(image_names, desc="Processing images"):
    input_path = os.path.join(image_folder, img_name)
    output_path = os.path.join(output_folder, os.path.splitext(img_name)[0] + ".txt")

    if not os.path.exists(input_path):
        print(f"⚠️ Skipping {img_name}: file not found")
        continue
    if os.path.exists(output_path):
        continue

    img = Image.open(input_path).convert("L")
    img_bin = binarization.nlbin(img)

    seg_result = pageseg.segment(img_bin)

    if not seg_result or (isinstance(seg_result, dict) and "lines" not in seg_result):
        print(f"⚠️ Skipping {img_name}: segmentation empty or invalid")
        continue

    try:
        predictions = [rec.prediction for rec in rpred.rpred(model, img_bin, seg_result)]
    except Exception as e:
        print(f"❌ Error processing {img_name}: {e}")
        continue

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(predictions))

Using device: cuda
Found 1004 images to process.


Processing images:   1%|          | 9/1004 [00:13<19:49,  1.20s/it]

⚠️ Skipping image_105_3.jpg: segmentation empty or invalid


Processing images:   2%|▏         | 24/1004 [00:38<18:54,  1.16s/it]

⚠️ Skipping image_113_1.jpg: segmentation empty or invalid


...

Processing images: 100%|█████████▉| 999/1004 [15:16<00:05,  1.11s/it]

⚠️ Skipping image_340_1.jpg: segmentation empty or invalid


Processing images: 100%|█████████▉| 1001/1004 [15:17<00:02,  1.11it/s]

⚠️ Skipping image_341_2.jpg: segmentation empty or invalid


Processing images: 100%|██████████| 1004/1004 [15:21<00:00,  1.09it/s]
